# ColQwen2 300-page inference

**Purpose:** clean rerun of the 300-page ColQwen2 inference pipeline on Google Colab T4.

This version:
- uses the flattened All_Images folder instead of the nested Images/ tree;
- keeps one stable page_id per image using the filename stem;
- indexes the directory in one `model.index(...) call instead of repeatedly rebuilding the index page-by-page;
- uses `store_collection_with_index=False to reduce index storage;
- retrieves **top-20** for all 50 questions so the same output can be reused for nDCG@5 and later analysis;
- saves rankings, scores, metadata, and timing to Drive;
- includes sanity checks so page/question mappings cannot silently drift.

In [11]:
# =========================
# 1. Mount Drive + packages
# =========================
from google.colab import drive
drive.mount("/content/drive")

import os
# Set before importing torch so CUDA allocator behavior is picked up.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# Upgrading torchao alongside other packages to resolve the peft compatibility issue
!pip -q install -U byaldi pandas numpy pillow tqdm torchao


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# =========================
# 2. Imports + T4 setup
# =========================
import os, glob, json, time, shutil, gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm

from byaldi import RAGMultiModalModel

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")
else:
    raise RuntimeError("A CUDA GPU is required. In Colab: Runtime -> Change runtime type -> T4 GPU.")

device = "cuda"
torch.cuda.empty_cache()

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


Torch: 2.13.0+cu130
CUDA available: True
GPU: Tesla T4
GPU memory: 14.6 GB


## 3. Paths



In [3]:
# =========================
# 3. Project paths
# =========================
ROOT = Path("/content/drive/MyDrive/ColPali IR project")

# Existing data layout from the previous notebook
NESTED_IMAGES = ROOT / "Data" / "Pages" / "Images"
ALL_IMAGES = NESTED_IMAGES / "All_Images"

# Preferred prepared evaluation files
PREPARED = ROOT / "prepared_data"
Q_PATH = PREPARED / "queries_clean.csv"
QRELS_PATH = PREPARED / "qrels.csv"

# Fallback query file used by the older notebook
LEGACY_Q_PATH = ROOT / "Data" / "Queries" / "colpali_50_queries.csv"

INDEX_NAME = "colqwen2_300pages_all_images_t4"
INDEX_ROOT = ROOT / "byaldi_indexes"

RUN_DIR = ROOT / "inference_results"
RUN_DIR.mkdir(parents=True, exist_ok=True)
INDEX_ROOT.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("Nested images:", NESTED_IMAGES)
print("Flattened images:", ALL_IMAGES)
print("Prepared data exists:", PREPARED.exists())

ROOT: /content/drive/MyDrive/ColPali IR project
Nested images: /content/drive/MyDrive/ColPali IR project/Data/Pages/Images
Flattened images: /content/drive/MyDrive/ColPali IR project/Data/Pages/Images/All_Images
Prepared data exists: True


In [4]:
# =========================
# 4. Build / validate All_Images
# =========================
ALL_IMAGES.mkdir(parents=True, exist_ok=True)

nested_pngs = sorted(
    p for p in NESTED_IMAGES.rglob("*.png")
    if p.parent != ALL_IMAGES
)

flat_pngs = sorted(ALL_IMAGES.glob("*.png"))

print(f"Nested PNG pages found: {len(nested_pngs)}")
print(f"Already flattened PNG pages: {len(flat_pngs)}")

# Copy only missing files. Filename stems are the stable page IDs.
existing_names = {p.name for p in ALL_IMAGES.glob("*.png")}

copied = 0
for src in tqdm(nested_pngs, desc="Flattening pages"):
    if src.name not in existing_names:
        shutil.copy2(src, ALL_IMAGES / src.name)
        copied += 1

image_paths = sorted(ALL_IMAGES.glob("*.png"))

print(f"Copied this run: {copied}")
print(f"Total images in All_Images: {len(image_paths)}")

if len(image_paths) < 300:
    raise ValueError(f"Expected about 300 pages, but found only {len(image_paths)} in All_Images.")

if len(image_paths) != len({p.stem for p in image_paths}):
    raise ValueError("Duplicate page stems detected. Every page must have a unique page_id.")

print("First 5:", [p.name for p in image_paths[:5]])
print("Last 5:", [p.name for p in image_paths[-5:]])

Nested PNG pages found: 310
Already flattened PNG pages: 0


Flattening pages:   0%|          | 0/310 [00:00<?, ?it/s]

Copied this run: 310
Total images in All_Images: 310
First 5: ['paper_01_page_001.png', 'paper_01_page_002.png', 'paper_01_page_003.png', 'paper_01_page_004.png', 'paper_01_page_005.png']
Last 5: ['paper_14_page_013.png', 'paper_14_page_014.png', 'paper_14_page_015.png', 'paper_14_page_016.png', 'paper_14_page_017.png']


In [5]:
# =========================
# 5. Build stable page metadata
# =========================
pages = pd.DataFrame({
    "doc_id": np.arange(len(image_paths), dtype=int),
    "page_id": [p.stem for p in image_paths],
    "image_path": [str(p) for p in image_paths],
})

pages.to_csv(RUN_DIR / "indexed_pages.csv", index=False)

display(pages.head())
print("Page count:", len(pages))

,doc_id,page_id,image_path
0,0,paper_01_page_001,/content/drive/MyDrive/ColPali IR project/Data...
1,1,paper_01_page_002,/content/drive/MyDrive/ColPali IR project/Data...
2,2,paper_01_page_003,/content/drive/MyDrive/ColPali IR project/Data...
3,3,paper_01_page_004,/content/drive/MyDrive/ColPali IR project/Data...
4,4,paper_01_page_005,/content/drive/MyDrive/ColPali IR project/Data...


Page count: 310


## 6. Load ColQwen2 once



In [9]:
# =========================
# 6. Load ColQwen2
# =========================
torch.cuda.empty_cache()
gc.collect()

# Suppress warning messages regarding HF_TOKEN if you don't have one set up
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

model = RAGMultiModalModel.from_pretrained(
    "vidore/colqwen2-v1.0",
    device=device,
)

print("ColQwen2 loaded successfully.")

Verbosity is set to 1 (active). Pass verbose=0 to make quieter.


Loading weights:   0%|          | 0/731 [00:00<?, ?it/s]

W1117 23:02:34.152000 766 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 74.0MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

... (8 outputs left out)

added_tokens.json:   0%|          | 0.00/392 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

ColQwen2 loaded successfully.


## 7. Index all 300 pages

This is the main improvement over the old page-by-page loop.

Byaldi supports indexing a directory containing multiple image files. store_collection_with_index=False avoids storing base64 copies of the images inside the index.

In [12]:
# =========================
# 7. 300-page indexing
# =========================
index_path = INDEX_ROOT / INDEX_NAME

# Re-running this cell with overwrite=True rebuilds the index.
# Keep True for the clean final run; change to False if you want to preserve an existing index.
REBUILD_INDEX = True

start = time.perf_counter()

# Note: index_root is removed because it is not accepted by model.index().
model.index(
    input_path=str(ALL_IMAGES),
    index_name=INDEX_NAME,
    doc_ids=pages["doc_id"].tolist(),
    metadata=[
        {
            "page_id": row.page_id,
            "image_path": row.image_path,
        }
        for row in pages.itertuples(index=False)
    ],
    store_collection_with_index=False,
    overwrite=REBUILD_INDEX,
)

index_seconds = time.perf_counter() - start

print(f"Indexed {len(pages)} pages")
print(f"Indexing time: {index_seconds:.2f} s")
print(f"Indexing time: {index_seconds/60:.2f} min")
print(f"Average: {index_seconds/len(pages):.3f} s/page")

# Save timing separately for the report.
with open(RUN_DIR / "index_timing.json", "w") as f:
    json.dump({
        "pages": int(len(pages)),
        "seconds": float(index_seconds),
        "seconds_per_page": float(index_seconds / len(pages)),
        "estimated_300_pages_seconds": float(index_seconds),
    }, f, indent=2)

Indexing file: /content/drive/MyDrive/ColPali IR project/Data/Pages/Images/All_Images/paper_01_page_001.png
Added page 1 of document 0 to index.
Index exported to .byaldi/colqwen2_300pages_all_images_t4
Indexing file: /content/drive/MyDrive/ColPali IR project/Data/Pages/Images/All_Images/paper_01_page_002.png
Added page 1 of document 1 to index.
Index exported to .byaldi/colqwen2_300pages_all_images_t4
Indexing file: /content/drive/MyDrive/ColPali IR project/Data/Pages/Images/All_Images/paper_01_page_003.png
Added page 1 of document 2 to index.
Index exported to .byaldi/colqwen2_300pages_all_images_t4
Indexing file: /content/drive/MyDrive/ColPali IR project/Data/Pages/Images/All_Images/paper_01_page_004.png
Added page 1 of document 3 to index.
Index exported to .byaldi/colqwen2_300pages_all_images_t4
Indexing file: /content/drive/MyDrive/ColPali IR project/Data/Pages/Images/All_Images/paper_01_page_005.png
Added page 1 of document 4 to index.
Index exported to .byaldi/colqwen2_300pages

## 8. Load the 50-question evaluation set



In [13]:
# =========================
# 8. Load queries
# =========================
if Q_PATH.exists():
    q = pd.read_csv(Q_PATH)
    print("Loaded:", Q_PATH)
else:
    q = pd.read_csv(LEGACY_Q_PATH)
    print("Loaded legacy:", LEGACY_Q_PATH)

# Normalize column names from either schema.
rename_map = {}
if "questions" in q.columns and "question" not in q.columns:
    rename_map["questions"] = "question"

q = q.rename(columns=rename_map)

required = {"question"}
missing = required - set(q.columns)
if missing:
    raise ValueError(f"Query file is missing required columns: {missing}")

if "qid" not in q.columns:
    q["qid"] = [f"q{i+1:03d}" for i in range(len(q))]

# Normalize question type labels.
if "question_type" not in q.columns:
    q["question_type"] = "unknown"

q["question_type"] = q["question_type"].astype(str).str.lower().str.strip()

# If the prepared file has page_id, keep it.
# Otherwise derive page_id from paper + page_no.
if "page_id" not in q.columns:
    if {"paper", "page_no"}.issubset(q.columns):
        q["paper"] = q["paper"].astype(str).str.replace(".pdf", "", regex=False)
        q["page_id"] = (
            q["paper"].str.replace(".pdf", "", regex=False)
            + "_page_"
            + q["page_no"].astype(int).astype(str).str.zfill(3)
        )
    else:
        raise ValueError("Cannot construct page_id from the query file.")

q["qid"] = q["qid"].astype(str)
q["page_id"] = q["page_id"].astype(str)
q["question"] = q["question"].astype(str)

if len(q) != 50:
    print(f"WARNING: expected 50 questions, found {len(q)}")

missing_gold = sorted(set(q["page_id"]) - set(pages["page_id"]))
if missing_gold:
    raise ValueError(
        "Some gold page_ids are not present in All_Images. "
        f"Examples: {missing_gold[:10]}"
    )

print("Questions:", len(q))
print(q["question_type"].value_counts(dropna=False))
display(q.head())

Loaded: /content/drive/MyDrive/ColPali IR project/prepared_data/queries_clean.csv
Questions: 50
question_type
text      18
table     17
figure    15
Name: count, dtype: int64


,qid,question,question_type,page_id
0,q001,What role does dropout play in the unsupervise...,text,paper_01_page_002
1,q002,Which dropout probability gives the strongest ...,table,paper_01_page_004
2,q003,Why are entailment pairs from SNLI and MNLI us...,text,paper_01_page_005
3,q004,How does first-last average pooling compare wi...,table,paper_01_page_008
4,q005,Why does SPLADE retain advantages of tradition...,text,paper_02_page_002


In [14]:
# =========================
# 9. Retrieval function
# =========================
def result_page_id(result):
    meta = result.get("metadata") or {}
    if isinstance(meta, dict) and meta.get("page_id") is not None:
        return str(meta["page_id"])
    return None

def retrieve_all(model, queries_df, k=20):
    rows = []

    start = time.perf_counter()

    for row in tqdm(queries_df.itertuples(index=False), total=len(queries_df), desc=f"Retrieving top-{k}"):
        results = model.search(str(row.question), k=k)

        for rank, r in enumerate(results, start=1):
            rows.append({
                "qid": str(row.qid),
                "rank": rank,
                "doc_id": int(r["doc_id"]),
                "page_id": result_page_id(r),
                "score": float(r["score"]),
            })

    elapsed = time.perf_counter() - start
    return pd.DataFrame(rows), elapsed

## 10. Retrieve top-20 for all 50 questions

We deliberately save **top-20**, not just top-5. This gives us enough information for later analyses without rerunning the model. nDCG@5 will still be computed from the first five results.

In [16]:
# =========================
# 10. Retrieve top-20 (Fixed for 'Result' object attributes)
# =========================

def result_page_id_fixed(result):
    # Accessing metadata as an attribute of the Result object
    meta = getattr(result, "metadata", {})
    if isinstance(meta, dict) and meta.get("page_id") is not None:
        return str(meta["page_id"])
    return None

def retrieve_all_fixed(model, queries_df, k=20):
    rows = []
    start = time.perf_counter()

    for row in tqdm(queries_df.itertuples(index=False), total=len(queries_df), desc=f"Retrieving top-{k}"):
        results = model.search(str(row.question), k=k)

        for rank, r in enumerate(results, start=1):
            # Access fields as attributes instead of dictionary items
            rows.append({
                "qid": str(row.qid),
                "rank": rank,
                "doc_id": int(getattr(r, "doc_id")),
                "page_id": result_page_id_fixed(r),
                "score": float(getattr(r, "score")),
            })

    elapsed = time.perf_counter() - start
    return pd.DataFrame(rows), elapsed

# Run retrieval using the updated logic
retrieval_df, retrieval_seconds = retrieve_all_fixed(model, q, k=20)

if retrieval_df["page_id"].isna().any():
    bad = retrieval_df[retrieval_df["page_id"].isna()].head()
    print("WARNING: some results have no metadata page_id:")
    display(bad)

retrieval_df.to_csv(RUN_DIR / "colqwen2_maxsim_top20.csv", index=False)

with open(RUN_DIR / "retrieval_timing.json", "w") as f:
    json.dump({
        "queries": int(len(q)),
        "k": 20,
        "total_seconds": float(retrieval_seconds),
        "mean_seconds_per_query": float(retrieval_seconds / len(q)),
        "median_seconds_per_query": None
    }, f, indent=2)

print(f"Total retrieval time: {retrieval_seconds:.2f} s")
print(f"Mean query time: {retrieval_seconds/len(q):.3f} s/query")
display(retrieval_df.head(10))

Retrieving top-20:   0%|          | 0/50 [00:00<?, ?it/s]

Total retrieval time: 8.97 s
Mean query time: 0.179 s/query


,qid,rank,doc_id,page_id,score
0,q001,1,176,paper_10_page_018,19.625
1,q001,2,1,paper_01_page_002,19.375
2,q001,3,0,paper_01_page_001,19.125
3,q001,4,161,paper_10_page_003,19.125
4,q001,5,3,paper_01_page_004,19.125
5,q001,6,177,paper_10_page_019,19.000
6,q001,7,2,paper_01_page_003,18.625
7,q001,8,8,paper_01_page_009,18.500
8,q001,9,162,paper_10_page_004,18.375
9,q001,10,160,paper_10_page_002,17.750


## 11. Verify the ranking mapping

This catches the most dangerous evaluation bug: comparing Byaldi's integer `doc_id` to a filename/page ID or accidentally shifting page numbering.

In [17]:
# =========================
# 11. Mapping sanity checks
# =========================
doc_to_page = dict(zip(pages["doc_id"], pages["page_id"]))

retrieval_df["page_id_from_docid"] = retrieval_df["doc_id"].map(doc_to_page)

mapping_ok = (
    retrieval_df["page_id"].notna()
    & retrieval_df["page_id_from_docid"].notna()
    & (retrieval_df["page_id"] == retrieval_df["page_id_from_docid"])
)

print(f"DocID -> page_id mapping correct: {mapping_ok.mean()*100:.2f}%")

if not mapping_ok.all():
    display(retrieval_df.loc[~mapping_ok].head(20))
    raise ValueError("Retrieval metadata mapping is inconsistent.")

# Every query should have exactly 20 rows.
counts = retrieval_df.groupby("qid").size()
if not (counts == 20).all():
    raise ValueError("Some queries do not have exactly 20 retrieved results.")

print("All 50 queries have 20 ranked results.")

DocID -> page_id mapping correct: 100.00%
All 50 queries have 20 ranked results.


## 12. nDCG@5

For the custom testbed each question has one gold page, so binary relevance is appropriate. The score is calculated from the retrieved ranking and the page_id gold label, not from the arbitrary Byaldi doc_id.

In [18]:
# =========================
# 12. nDCG@5
# =========================
def ndcg_single_gold(ranked_page_ids, gold_page_id, k=5):
    rels = np.array(
        [1.0 if pid == gold_page_id else 0.0 for pid in ranked_page_ids[:k]]
    )
    discounts = 1.0 / np.log2(np.arange(2, len(rels) + 2))
    dcg = float(np.sum(rels * discounts))

    # One relevant document => ideal DCG is always 1 at k>=1.
    return dcg

top5 = (
    retrieval_df.sort_values(["qid", "rank"])
    .groupby("qid")
    .head(5)
)

qid_to_gold = dict(zip(q["qid"], q["page_id"]))

per_query = []
for qid, g in top5.groupby("qid"):
    ranked = g.sort_values("rank")["page_id"].tolist()
    gold = qid_to_gold[qid]
    score = ndcg_single_gold(ranked, gold, k=5)
    per_query.append({"qid": qid, "gold_page_id": gold, "ndcg@5": score})

per_query_df = pd.DataFrame(per_query).merge(
    q[["qid", "question", "question_type"]],
    on="qid",
    how="left"
)

overall_ndcg5 = per_query_df["ndcg@5"].mean()

print(f"ColQwen2 MaxSim nDCG@5 = {overall_ndcg5:.4f}")
display(per_query_df.head(10))

ColQwen2 MaxSim nDCG@5 = 0.6620


,qid,gold_page_id,ndcg@5,question,question_type
0,q001,paper_01_page_002,0.630930,What role does dropout play in the unsupervise...,text
1,q002,paper_01_page_004,1.000000,Which dropout probability gives the strongest ...,table
2,q003,paper_01_page_005,0.630930,Why are entailment pairs from SNLI and MNLI us...,text
3,q004,paper_01_page_008,1.000000,How does first-last average pooling compare wi...,table
4,q005,paper_02_page_002,0.430677,Why does SPLADE retain advantages of tradition...,text
5,q006,paper_02_page_004,0.630930,Which listed method has the strongest MS MARCO...,table
6,q007,paper_03_page_002,0.500000,How does DSI turn document retrieval into a te...,text
7,q008,paper_03_page_006,0.630930,Why can semantically structured document ident...,text
8,q009,paper_03_page_008,0.630930,Which DSI variant performs best in the documen...,table
9,q010,paper_03_page_009,1.000000,How does DSI performance change as model size ...,figure


In [22]:
# =========================
# 13. nDCG@5 by question type
# =========================
by_type = (
    per_query_df
    .groupby("question_type")["ndcg@5"]
    .agg(["mean", "count"])
    .reset_index()
    .rename(columns={"mean": "nDCG@5", "count": "queries"})
)

print("Overall:", round(overall_ndcg5, 4))
display(by_type)

per_query_df.to_csv(RUN_DIR / "colqwen2_maxsim_per_query_ndcg5.csv", index=False)
by_type.to_csv(RUN_DIR / "colqwen2_maxsim_ndcg5_by_type.csv", index=False)

# Using iterrows() preserves the original column name strings as dictionary keys
summary = {
    "system": "ColQwen2 MaxSim",
    "model": "vidore/colqwen2-v1.0",
    "pages": int(len(pages)),
    "queries": int(len(q)),
    "nDCG@5": float(overall_ndcg5),
    "nDCG@5_by_type": {
        str(row["question_type"]): float(row["nDCG@5"])
        for _, row in by_type.iterrows()
    },
}
with open(RUN_DIR / "colqwen2_maxsim_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Saved results to:", RUN_DIR)

Overall: 0.662


,question_type,nDCG@5,queries
0,figure,0.726186,15
1,table,0.575622,17
2,text,0.690035,18


Saved results to: /content/drive/MyDrive/ColPali IR project/inference_results


## 14. Efficiency report

The 20-page timing from the old notebook is retained only for comparison. This cell measures the actual 300-page flattened run and reports the quantities needed in the project report.

In [23]:
# =========================
# 14. Efficiency summary
# =========================
print("=== ColQwen2 300-page T4 run ===")
print(f"Pages:                  {len(pages)}")
print(f"Indexing time:          {index_seconds:.2f} s ({index_seconds/60:.2f} min)")
print(f"Indexing time/page:     {index_seconds/len(pages):.3f} s")
print(f"Estimated 300 pages:    {index_seconds/60:.2f} min")
print(f"50-query retrieval:     {retrieval_seconds:.2f} s")
print(f"Mean retrieval/query:   {retrieval_seconds/len(q):.3f} s")
print(f"Overall nDCG@5:         {overall_ndcg5:.4f}")

=== ColQwen2 300-page T4 run ===
Pages:                  310
Indexing time:          1482.49 s (24.71 min)
Indexing time/page:     4.782 s
Estimated 300 pages:    24.71 min
50-query retrieval:     8.97 s
Mean retrieval/query:   0.179 s
Overall nDCG@5:         0.6620


In [25]:
import shutil

# Source local index path created by Byaldi
local_index_path = Path(".byaldi") / INDEX_NAME

# Destination path on Google Drive
drive_index_path = INDEX_ROOT / INDEX_NAME

if local_index_path.exists():
    print(f"Copying index from local directory to Google Drive...")
    if drive_index_path.exists():
        print(f"Removing existing index at {drive_index_path} to overwrite.")
        shutil.rmtree(drive_index_path)
    shutil.copytree(local_index_path, drive_index_path)
    print(f"Successfully saved index to: {drive_index_path}")
else:
    print(f"Local index directory not found at {local_index_path}. Please verify if the index was generated.")

Copying index from local directory to Google Drive...
Removing existing index at /content/drive/MyDrive/ColPali IR project/byaldi_indexes/colqwen2_300pages_all_images_t4 to overwrite.
Successfully saved index to: /content/drive/MyDrive/ColPali IR project/byaldi_indexes/colqwen2_300pages_all_images_t4


In [26]:
# Display the computed nDCG@5 results by question type
display(by_type)

,question_type,nDCG@5,queries
0,figure,0.726186,15
1,table,0.575622,17
2,text,0.690035,18


### Important note for the report

This notebook evaluates the **custom 300-page scientific-paper testbed**. Do not use its nDCG@5 as if it were the paper's ViDoRe benchmark number. Keep the paper reproduction benchmark and the custom testbed results as separate experiments.

The model is vidore/colqwen2-v1.0; Byaldi supports this checkpoint and supports indexing a directory of images directly.